# state_head_fold_stability_guard_v1

Purpose: test whether the state-head target lifts from `rebound_state_head_lift_guard_v1` are stable across fold-native labels, before using them as a PPO auxiliary-state recipe.

Governance:
- Baseline: `rebound_state_head_lift_guard_v1`.
- Role: diagnose / overfit guard.
- Changed layer: model-selection target validation; no runtime execution change.
- Target gate remains `+30pp` relative return and `<=15%` MaxDD after future training.
- Accepted trade-off: block a promising lift recipe if the non-trivial lift rows are label-unstable across walk-forward folds.

Closest historical success:
- `rebound_representation_separability_v1` showed `momentum_drawdown` features can separate rebound states across folds.

Closest historical failure:
- `state_objective_state_conditioned_loss_v1` fit the sparse rebound rows in-sample but failed fold-heldout rebound accuracy, so the current risk is target memorization / label instability.

External references:
- Qlib reproducible quant workflow: https://github.com/microsoft/qlib and https://arxiv.org/abs/2009.11189
- FinRL train/validation/test trading workflow: https://github.com/AI4Finance-Foundation/FinRL and https://arxiv.org/abs/2011.09607
- Probability of Backtest Overfitting / selection discipline: https://doi.org/10.21314/JCF.2016.322
- Supervised contrastive representation learning: https://arxiv.org/abs/2004.11362


In [1]:

from __future__ import annotations

import json
import sys
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display


def find_repo_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / 'AGENTS.md').exists() and (candidate / 'src' / 'soxl_quant').exists():
            return candidate
    raise RuntimeError(f'Cannot locate repo root from {start}')


REPO_ROOT = find_repo_root(Path.cwd().resolve())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.soxl_quant.app.representation_diagnostics import (
    StateHeadLiftStabilityConfig,
    StateHeadTargetRecipeConfig,
    score_state_head_lift_stability,
    score_state_head_target_recipes,
)
from src.soxl_quant.policy.policy_builder import state_objective_target_position_from_row

EXPERIMENT = 'state_head_fold_stability_guard_v1'
OUT_DIR = REPO_ROOT / 'runtime' / 'model_selection' / 'card' / EXPERIMENT
OUT_DIR.mkdir(parents=True, exist_ok=True)

TARGETS_PATH = REPO_ROOT / 'runtime' / 'model_selection' / 'card' / 'state_objective_fold_native_teacher_router_v1' / 'fold_native_teacher_targets.csv'
LIFT_GUARD_SELECTION = REPO_ROOT / 'runtime' / 'model_selection' / 'card' / 'rebound_state_head_lift_guard_v1' / 'selection.json'
RECIPE_SELECTION = REPO_ROOT / 'runtime' / 'model_selection' / 'card' / 'rebound_state_head_target_recipe_v1' / 'selection.json'

with LIFT_GUARD_SELECTION.open() as fh:
    lift_guard_selection = json.load(fh)
with RECIPE_SELECTION.open() as fh:
    recipe_selection = json.load(fh)

targets = pd.read_csv(TARGETS_PATH)
targets['datetime'] = pd.to_datetime(targets['datetime'])
targets['fold'] = targets['fold'].astype(str)
targets['is_rebound_cluster'] = targets['is_rebound_cluster'].fillna(False).astype(bool)
targets['is_downtrend_state'] = targets['is_downtrend_state'].fillna(False).astype(bool)
targets['true_target_position'] = [state_objective_target_position_from_row(row) for row in targets.to_dict('records')]
targets['default_target_position'] = np.where(
    targets['is_downtrend_state'],
    pd.to_numeric(targets['risk_target_position'], errors='coerce'),
    pd.to_numeric(targets['teacher_position'], errors='coerce'),
)

base_context = {
    'experiment': EXPERIMENT,
    'created_at': datetime.now(timezone.utc).isoformat(),
    'source_targets': str(TARGETS_PATH.relative_to(REPO_ROOT)),
    'source_lift_guard': str(LIFT_GUARD_SELECTION.relative_to(REPO_ROOT)),
    'rows': int(len(targets)),
    'folds': sorted(targets['fold'].unique().tolist()),
    'rebound_rows': int(targets['is_rebound_cluster'].sum()),
    'baseline_feature_set': lift_guard_selection['best_feature_set'],
    'baseline_threshold': float(lift_guard_selection['best_threshold']),
    'baseline_effective_false_positive': int(lift_guard_selection['best_effective_false_positive']),
    'baseline_effective_false_positive_rate': float(lift_guard_selection['best_effective_false_positive_rate']),
}
print(json.dumps(base_context, indent=2, sort_keys=True))


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


{
  "baseline_effective_false_positive": 1,
  "baseline_effective_false_positive_rate": 0.0007974481658692185,
  "baseline_feature_set": "momentum_drawdown",
  "baseline_threshold": 0.5,
  "created_at": "2026-05-25T14:20:56.682552+00:00",
  "experiment": "state_head_fold_stability_guard_v1",
  "folds": [
    "wf_2022_train",
    "wf_2023_2024_train"
  ],
  "rebound_rows": 7,
  "rows": 1261,
  "source_lift_guard": "runtime/model_selection/card/rebound_state_head_lift_guard_v1/selection.json",
  "source_targets": "runtime/model_selection/card/state_objective_fold_native_teacher_router_v1/fold_native_teacher_targets.csv"
}


In [2]:

feature_sets = {
    'momentum_drawdown': ['mom_5', 'mom_21', 'drawdown_63'],
}
recipe_config = StateHeadTargetRecipeConfig(
    min_recall=0.70,
    min_precision=0.05,
    max_false_positive_rate=0.20,
    min_effective_lift=0.01,
    max_effective_false_positive_rate=0.01,
    max_false_positive_lift_mean=0.01,
    max_false_positive_lift_max=0.20,
    max_target_mae=0.02,
    max_rebound_mae=0.03,
    min_covered_folds=2,
)
recipe_result = score_state_head_target_recipes(
    targets,
    label_col='is_rebound_cluster',
    fold_col='fold',
    true_target_col='true_target_position',
    default_target_col='default_target_position',
    state_target_col='rebound_target_position',
    feature_sets=feature_sets,
    thresholds=[float(lift_guard_selection['best_threshold'])],
    min_positive_train=2,
    min_positive_eval=2,
    config=recipe_config,
)

row_scores = recipe_result.row_scores.merge(
    targets.reset_index().rename(columns={'index': 'row_index'})[
        [
            'row_index', 'datetime', 'is_downtrend_state', 'mom_5', 'mom_21',
            'drawdown_63', 'rv_21', 'soxl_soxx_ratio_widen_21', 'soxx_mom_21',
        ]
    ],
    on='row_index',
    how='left',
)

stability = score_state_head_lift_stability(
    row_scores,
    group_col='datetime',
    label_col='is_rebound_cluster',
    config=StateHeadLiftStabilityConfig(min_lift=0.01, max_unstable_effective_lift_rate=0.0),
)

group_metrics = stability.group_metrics
scored_rows = stability.row_scores
unstable_effective_rows = scored_rows[scored_rows['unstable_effective_lift']].copy()
unstable_groups = group_metrics[group_metrics['label_unstable']].copy()

# Keep sidecars compact: group-level metrics and only the rows that matter for the decision.
group_metrics.to_csv(OUT_DIR / 'lift_stability_group_metrics.csv', index=False)
unstable_effective_rows.to_csv(OUT_DIR / 'unstable_effective_lift_rows.csv', index=False)
unstable_groups.to_csv(OUT_DIR / 'unstable_label_groups.csv', index=False)

print('recipe score')
display(recipe_result.recipe_scores)
print('stability summary')
print(json.dumps(stability.summary, indent=2, sort_keys=True))
print('unstable effective lift rows')
display(unstable_effective_rows[[
    'row_index', 'datetime', 'fold', 'is_rebound_cluster', 'is_downtrend_state',
    'state_score', 'default_target_position', 'rebound_target_position', 'target_lift',
    'mom_5', 'mom_21', 'drawdown_63', 'soxl_soxx_ratio_widen_21', 'soxx_mom_21',
]])


recipe score


,feature_set,threshold,samples,positives,negatives,covered_folds,true_positive,false_positive,false_negative,true_negative,...,false_positive_rate,effective_false_positive,effective_false_positive_rate,false_positive_lift_mean,false_positive_lift_max,target_mae,rebound_target_mae,non_rebound_target_mae,recipe_gate,recipe_loss
0,momentum_drawdown,0.5,1261,7,1254,2,7,123,0,1131,...,0.098086,1,0.000797,0.001462,0.179855,0.000301,0.028571,0.000143,True,0.0


stability summary
{
  "effective_lift_rows": 5,
  "false_positive_effective_lift_rows": 1,
  "groups": 756,
  "positive_effective_lift_rows": 4,
  "rows": 1261,
  "stability_gate": false,
  "unstable_effective_lift_rate": 1.0,
  "unstable_effective_lift_rows": 5,
  "unstable_false_positive_effective_lift_rows": 1,
  "unstable_groups": 7
}
unstable effective lift rows


,row_index,datetime,fold,is_rebound_cluster,is_downtrend_state,state_score,default_target_position,rebound_target_position,target_lift,mom_5,mom_21,drawdown_63,soxl_soxx_ratio_widen_21,soxx_mom_21
210,105,2020-06-03,wf_2022_train,True,True,1.394984,0.250000,0.450000,0.200000,0.149455,0.542637,-0.166154,0.320821,0.167938
211,610,2020-06-03,wf_2023_2024_train,False,True,1.213970,0.250000,0.429855,0.179855,0.149455,0.542637,-0.166154,0.320821,0.167938
631,820,2021-04-05,wf_2023_2024_train,True,False,1.810458,0.200000,0.350000,0.150000,0.193951,0.501001,-0.033962,0.288829,0.164624
633,821,2021-04-06,wf_2023_2024_train,True,False,0.917593,0.200000,0.350000,0.150000,0.212070,0.324580,-0.068739,0.184696,0.118076
635,822,2021-04-07,wf_2023_2024_train,True,False,2.261880,0.208429,0.350000,0.141571,0.221284,0.576308,-0.068739,0.334642,0.181072


In [3]:

stability_gate = bool(stability.summary['stability_gate'])
if stability_gate:
    decision = 'state_head_lift_stable_pre_ppo_handoff_allowed'
    next_mechanism = 'fold_local_ppo_aux_rebound_state_head_with_drawdown_guard'
    training_allowed = True
    blocker = ''
else:
    decision = 'state_head_lift_fold_unstable_no_ppo_handoff'
    next_mechanism = 'fold_consensus_rebound_target_generation_before_aux_ppo'
    training_allowed = False
    blocker = 'All non-trivial target lifts occur on fold-label-unstable dates; using them directly would train on unstable sparse labels.'

summary = {
    'base_context': base_context,
    'recipe_summary': recipe_result.summary,
    'stability_summary': stability.summary,
    'decision': decision,
    'next_mechanism': next_mechanism,
    'training_allowed_from_current_lift_labels': training_allowed,
    'promotion_allowed': False,
    'full_gate': False,
    'blocker': blocker,
    'conclusion': [
        f"Effective target-lift rows: {stability.summary['effective_lift_rows']}.",
        f"Unstable effective target-lift rows: {stability.summary['unstable_effective_lift_rows']} ({stability.summary['unstable_effective_lift_rate']:.3f}).",
        f"Unstable false-positive effective lifts: {stability.summary['unstable_false_positive_effective_lift_rows']}.",
        f"Decision: {decision}; next mechanism: {next_mechanism}.",
        'No runtime promotion; this is an overfit guard on target labels before any auxiliary state-head PPO training.',
    ],
}
selection = {
    'experiment': EXPERIMENT,
    'created_at': base_context['created_at'],
    'source_experiment': 'rebound_state_head_lift_guard_v1',
    'direct_comparison_baseline': 'rebound_state_head_target_recipe_v1',
    'decision': decision,
    'next_mechanism': next_mechanism,
    'training_allowed_from_current_lift_labels': training_allowed,
    'promotion_allowed': False,
    'full_gate': False,
    'stability_gate': stability_gate,
    'blocker': blocker,
    'effective_lift_rows': int(stability.summary['effective_lift_rows']),
    'unstable_effective_lift_rows': int(stability.summary['unstable_effective_lift_rows']),
    'unstable_effective_lift_rate': float(stability.summary['unstable_effective_lift_rate']),
    'false_positive_effective_lift_rows': int(stability.summary['false_positive_effective_lift_rows']),
    'unstable_false_positive_effective_lift_rows': int(stability.summary['unstable_false_positive_effective_lift_rows']),
    'acceptance_gate_for_next_run': {
        'fold_consensus_or_stable_lift_labels_required': True,
        'relative_return_min_pp': 30.0,
        'max_dd_max_pct': 15.0,
        'fixed_split_required': True,
        'walkforward_required': True,
        'no_runtime_promotion_from_recipe_only': True,
    },
    'external_sources': [
        'https://github.com/microsoft/qlib',
        'https://arxiv.org/abs/2009.11189',
        'https://github.com/AI4Finance-Foundation/FinRL',
        'https://arxiv.org/abs/2011.09607',
        'https://doi.org/10.21314/JCF.2016.322',
        'https://arxiv.org/abs/2004.11362',
    ],
    'sidecars': [
        'base_context.json',
        'lift_stability_group_metrics.csv',
        'unstable_effective_lift_rows.csv',
        'unstable_label_groups.csv',
        'summary.json',
    ],
}

(OUT_DIR / 'base_context.json').write_text(json.dumps(base_context, indent=2, sort_keys=True, default=str) + '\n')
(OUT_DIR / 'selection.json').write_text(json.dumps(selection, indent=2, sort_keys=True, default=str) + '\n')
(OUT_DIR / 'summary.json').write_text(json.dumps(summary, indent=2, sort_keys=True, default=str) + '\n')
print(json.dumps(selection, indent=2, sort_keys=True))


{
  "acceptance_gate_for_next_run": {
    "fixed_split_required": true,
    "fold_consensus_or_stable_lift_labels_required": true,
    "max_dd_max_pct": 15.0,
    "no_runtime_promotion_from_recipe_only": true,
    "relative_return_min_pp": 30.0,
    "walkforward_required": true
  },
  "blocker": "All non-trivial target lifts occur on fold-label-unstable dates; using them directly would train on unstable sparse labels.",
  "created_at": "2026-05-25T14:20:56.682552+00:00",
  "decision": "state_head_lift_fold_unstable_no_ppo_handoff",
  "direct_comparison_baseline": "rebound_state_head_target_recipe_v1",
  "effective_lift_rows": 5,
  "experiment": "state_head_fold_stability_guard_v1",
  "external_sources": [
    "https://github.com/microsoft/qlib",
    "https://arxiv.org/abs/2009.11189",
    "https://github.com/AI4Finance-Foundation/FinRL",
    "https://arxiv.org/abs/2011.09607",
    "https://doi.org/10.21314/JCF.2016.322",
    "https://arxiv.org/abs/2004.11362"
  ],
  "false_positive_eff

## Conclusion

This notebook is the single-run conclusion for the state-head lift stability guard. The state-head recipe still detects the intended rebound-like states, but its non-trivial target lifts are not fold-label-stable. That makes the current lift labels unsafe as direct auxiliary PPO targets. The next model-selection step should generate fold-consensus or otherwise stable rebound target labels before any PPO training branch consumes this state head.